In [4]:
import json
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import os

load_dotenv()
user_agent = os.getenv("USER_AGENT")

HEADERS = {
    "User-Agent": user_agent,
    "Accept-Language": "en-US,en;q=0.5"
}


def product_json(soup):
    for tag in soup.select('script[type=\"application/ld+json\"]'):
        try:
            for item in json.loads(tag.string):
                if item.get('@type') == 'Product': return item
        except (TypeError, ValueError): pass
    return {}

def get_title(soup):
    return product_json(soup).get('name', '')

def get_price(soup):
    m = re.search(r'\"ppd\":\\{.*?\"finalPrice\":(\d+(?:\.\d+)?)', str(soup), re.S)
    if m: return f'₹{float(m.group(1)):,.0f}'
    price = product_json(soup).get('offers', {}).get('price')
    return f'₹{float(price):,.0f}' if price is not None else ''

def get_rating(soup):
    rating = (product_json(soup).get('aggregateRating') or {}).get('ratingValue')
    if rating is not None: return str(rating)
    m = re.search(r'\"pr\":\\{.*?\"rating\":([0-9.]+)', str(soup), re.S)
    return m.group(1) if m and float(m.group(1)) > 0 else ''


if __name__ == "__main__":

    url = "https://www.flipkart.com/search?q=gaming+laptop"

    webpage = requests.get(
        url,
        headers=HEADERS,
        timeout=10
    )

    print("Status:", webpage.status_code)

    soup = BeautifulSoup(webpage.content, "html.parser")

    links = soup.select('a[href*="/p/"]')

    links_list = []

    for link in links:
        href = link.get("href")
        if href and href not in links_list:
            links_list.append(href)

    print("Links found:", len(links_list))

    data = {
        "title": [],
        "price": [],
        "rating": [],
        "link": []
    }

    for link in links_list[:10]:

        product_url = "https://www.flipkart.com" + link

        try:
            product_page = requests.get(
                product_url,
                headers=HEADERS,
                timeout=10
            )

            product_soup = BeautifulSoup(
                product_page.content,
                "html.parser"
            )

            data["title"].append(get_title(product_soup))
            data["price"].append(get_price(product_soup))
            data["rating"].append(get_rating(product_soup))
            data["link"].append(product_url)

        except requests.exceptions.RequestException as e:
            print("Request failed:", e)

    flipkart_df = pd.DataFrame(data)

    flipkart_df = flipkart_df[
        flipkart_df["title"] != ""
    ]

    file_name = "flipkart_data.csv"

    # Rebuild the file each run; append mode preserves stale malformed rows.
    flipkart_df.to_csv(file_name, index=False, encoding='utf-8-sig')

    print("\nFinal Data:")
    print(flipkart_df)
    

Status: 200
Links found: 29

Final Data:
                                               title     price rating  \
0  HP Victus Intel Core i7 14th Gen 14650HX - (16...  ₹275,547          
1  DELL Alienware 15 AMD Ryzen 7 260 - (16 GB/512...  ₹127,990    4.5   
2  MSI Katana 15 Intel Core i5 13th Gen 13420H - ...   ₹96,990    4.2   
3  HP Victus AI AMD Ryzen 7 Octa Core 260 - (16 G...  ₹149,990          
4  ASUS TUF Gaming A15 (2026) AMD Ryzen 7 170 - (...   ₹97,990    4.5   
5  HP HyperX Omen Intel Core 7 240H - (24 GB/1 TB...  ₹164,990          
6  Acer Aspire 7 Intel Core 5 210H - (16 GB/512 G...   ₹69,990    4.3   
7  Lenovo IdeaPad Gaming 3 Intel Core i5 11th Gen...   ₹71,990    4.3   
8  GIGABYTE A16 Intel Core i5 13th Gen 13420H - (...   ₹89,990          
9  DELL Alienware 15 AMD Ryzen 7 260 - (16 GB/512...  ₹127,990    4.5   

                                                link  
0  https://www.flipkart.com/hp-victus-intel-core-...  
1  https://www.flipkart.com/dell-alienware-15